# APA Stress in America — Data Exploration

Loads `data/APA_Stress_in_America_data.csv` and cleans it into a tidy DataFrame (`df`), one row per reported statistic.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

DATA_PATH = "data/APA_Stress_in_America_data.csv"
raw = pd.read_csv(DATA_PATH)
raw.shape

(125, 12)

In [3]:
raw.head()

,id,wave,topic,measure,group,age_range,value,unit,source_url,location,verification,notes
0,1,2013,Stress level,Average stress level past month (1-10),All adults,18+,5.1,mean_1_10,https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,Secondary (press coverage; APA PDF blocked our fetcher),NaN
1,2,2013,Stress level,Extreme stress (8-10),All adults,18+,21.0,pct,https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,Secondary (press coverage; APA PDF blocked our fetcher),NaN
2,3,2013,Teens,Average stress level during school year (1-10),Teens,13-17,5.8,mean_1_10,https://healthland.time.com/2014/02/11/americas-teens-outscore-adults-on-str...,article,Secondary (press coverage; APA PDF blocked our fetcher),NaN
3,4,2013,Teens,"Average stress level past month, Aug 2013 (1-10)",Teens,13-17,4.6,mean_1_10,https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,Secondary (press coverage; APA PDF blocked our fetcher),NaN
4,5,2013,Teens,Extreme stress (8-10) during school year,Teens,13-17,27.0,pct,https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,Secondary (press coverage; APA PDF blocked our fetcher),NaN


In [4]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            125 non-null    int64  
 1   wave          125 non-null    object 
 2   topic         125 non-null    object 
 3   measure       125 non-null    object 
 4   group         125 non-null    object 
 5   age_range     125 non-null    object 
 6   value         125 non-null    float64
 7   unit          125 non-null    object 
 8   source_url    125 non-null    object 
 9   location      125 non-null    object 
 10  verification  125 non-null    object 
 11  notes         35 non-null     object 
dtypes: float64(1), int64(1), object(10)
memory usage: 11.8+ KB


## Cleaning

- **wave** → ordered categorical, plus a numeric `year` (the `2016-17` wave maps to 2017, when the second fielding ended).
- **age_range** → numeric `age_min` / `age_max` (open-ended ranges like `65+` get `age_max = NaN`).
- **group_type** → buckets the `group` labels (overall, generation, age band, etc.) so like can be compared with like.
- **unit** → `pct` values stay on 0–100; `mean_1_10` values are on a 1–10 scale. Never plot the two on the same axis.
- **is_primary** → `True` when the figure comes straight from an APA report rather than press coverage.
- **page** → page number pulled from `location` where available.

In [5]:
df = raw.copy()

# Wave: ordered category + numeric year
wave_order = sorted(df["wave"].unique(), key=lambda w: int(str(w)[:4]))
df["wave"] = pd.Categorical(df["wave"].astype(str), categories=wave_order, ordered=True)
df["year"] = df["wave"].astype(str).map(lambda w: int(w[:4]) + 1 if "-" in w else int(w))

# Age range -> numeric bounds
ages = df["age_range"].str.extract(r"^(?P<age_min>\d+)(?:-(?P<age_max>\d+)|\+)$")
df["age_min"] = pd.to_numeric(ages["age_min"]).astype("Int64")
df["age_max"] = pd.to_numeric(ages["age_max"]).astype("Int64")

# Group type
GENERATIONS = ["Gen Z teens", "Gen Z", "Gen Z adults", "Millennials", "Gen X", "Boomers", "Older adults", "Matures"]
def classify_group(g):
    if g == "All adults":
        return "overall"
    if g == "Teens":
        return "teens"
    if g.startswith("Parents"):
        return "parents"
    if g.startswith("Ages "):
        return "age band"
    if g in GENERATIONS:
        return "generation"
    if g.startswith("Gen Z "):  # Gen Z female / male
        return "gender"
    return "other"  # e.g. constant vs non-constant social media checkers
df["group_type"] = df["group"].map(classify_group)

# Source quality and page
df["is_primary"] = df["verification"].eq("APA primary")
df["page"] = pd.to_numeric(df["location"].str.extract(r"p\.(\d+)")[0]).astype("Int64")

# Tidy up text columns
for col in ["topic", "group", "unit", "group_type"]:
    df[col] = df[col].astype("category")
df["notes"] = df["notes"].fillna("")

df = df[[
    "id", "wave", "year", "topic", "measure", "group", "group_type",
    "age_range", "age_min", "age_max", "value", "unit",
    "is_primary", "verification", "source_url", "location", "page", "notes",
]].set_index("id")
df.head()

,wave,year,topic,measure,group,group_type,age_range,age_min,age_max,value,unit,is_primary,verification,source_url,location,page,notes
id,,,,,,,,,,,,,,,,,
1,2013,2013,Stress level,Average stress level past month (1-10),All adults,overall,18+,18,<NA>,5.1,mean_1_10,False,Secondary (press coverage; APA PDF blocked our fetcher),https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,<NA>,
2,2013,2013,Stress level,Extreme stress (8-10),All adults,overall,18+,18,<NA>,21.0,pct,False,Secondary (press coverage; APA PDF blocked our fetcher),https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,<NA>,
3,2013,2013,Teens,Average stress level during school year (1-10),Teens,teens,13-17,13,17,5.8,mean_1_10,False,Secondary (press coverage; APA PDF blocked our fetcher),https://healthland.time.com/2014/02/11/americas-teens-outscore-adults-on-str...,article,<NA>,
4,2013,2013,Teens,"Average stress level past month, Aug 2013 (1-10)",Teens,teens,13-17,13,17,4.6,mean_1_10,False,Secondary (press coverage; APA PDF blocked our fetcher),https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,<NA>,
5,2013,2013,Teens,Extreme stress (8-10) during school year,Teens,teens,13-17,13,17,27.0,pct,False,Secondary (press coverage; APA PDF blocked our fetcher),https://www.athleticbusiness.com/more-news/only-37-percent-of-teens-combat-s...,article,<NA>,


In [6]:
df.dtypes

wave            category
year               int64
topic           category
measure           object
group           category
group_type      category
age_range         object
age_min            Int64
age_max            Int64
value            float64
unit            category
is_primary          bool
verification      object
source_url        object
location          object
page               Int64
notes             object
dtype: object

## Quick checks

In [7]:
# Any values outside their unit's valid range?
bad_pct = df[(df["unit"] == "pct") & ~df["value"].between(0, 100)]
bad_mean = df[(df["unit"] == "mean_1_10") & ~df["value"].between(1, 10)]
print(f"out-of-range pct: {len(bad_pct)}, out-of-range 1-10 means: {len(bad_mean)}")
print(f"missing values:\n{df.isna().sum()[df.isna().sum() > 0]}")

out-of-range pct: 0, out-of-range 1-10 means: 0
missing values:
age_max    41
page       25
dtype: int64


In [8]:
pd.crosstab(df["topic"], df["wave"], margins=True)

wave,2013,2016-17,2018,2019,2020,2023,All
topic,,,,,,,
Diagnosis (self-reported),0,0,6,0,0,4,10
Health-related stress,0,0,0,0,0,1,1
Loneliness,0,0,0,0,5,0,5
News & stress,0,0,0,6,0,0,6
Self-rated mental health,0,0,12,0,6,1,19
Social media,0,11,3,0,0,0,14
Stress level,2,0,7,10,6,5,30
Stress symptoms,0,0,2,0,0,0,2
Support,0,0,2,0,6,0,8


In [9]:
pd.crosstab(df["group"], df["wave"])

wave,2013,2016-17,2018,2019,2020,2023
group,,,,,,
Ages 18-34,0,0,0,1,0,3
Ages 35-44,0,0,0,1,0,2
Ages 45-64,0,0,0,1,0,2
Ages 65+,0,0,0,1,0,1
All adults,2,3,3,2,3,3
Boomers,0,3,5,2,4,0
Constant checkers,0,3,0,0,0,0
Gen X,0,3,5,2,4,0
Gen Z,0,0,8,0,0,0


## Handy subsets

- `pct` — percentage measures (0–100)
- `means` — 1–10 average stress ratings

In [10]:
pct = df[df["unit"] == "pct"]
means = df[df["unit"] == "mean_1_10"]
print(f"pct rows: {len(pct)}, mean_1_10 rows: {len(means)}")
means.pivot_table(index="measure", columns="wave", values="value", aggfunc="first", observed=True)

pct rows: 87, mean_1_10 rows: 38


wave,2013,2016-17,2018,2019,2020,2023
measure,,,,,,
Average overall stress level (1-10),NaN,5.3,NaN,NaN,NaN,NaN
Average stress level (1-10),NaN,NaN,4.9,4.9,NaN,6.0
"Average stress level (1-10), as restated in 2020 report",NaN,NaN,5.6,NaN,NaN,NaN
"Average stress level (1-10), by age band as restated in 2023 report",NaN,NaN,NaN,5.7,NaN,NaN
Average stress level during school year (1-10),5.8,NaN,NaN,NaN,NaN,NaN
Average stress level past month (1-10),5.1,NaN,NaN,NaN,5.0,NaN
"Average stress level past month, Aug 2013 (1-10)",4.6,NaN,NaN,NaN,NaN,NaN
Average stress related to technology (1-10),NaN,3.2,NaN,NaN,NaN,NaN
Stress level they believe is healthy (1-10),3.9,NaN,NaN,NaN,NaN,NaN
